# NeuroLens — Batch Processing

Process multiple videos into the prototype's own sample list. The current app does not read it: its samples come from `infra/make_sample.py`.

**Inputs:** Edit the two variables in the next cell:
- `VIDEO_PATHS` — list of video file paths
- `SAMPLES_JSON_PATH` — output path for the prototype's sample list


## 1. Configuration

Edit these variables before running.

Directory structure:
```
data/
├── videos/                              ← put your video files here
├── output/
│   └── sample_fishermanfriend_2021/     ← per-video output folder
│       └── thumb.jpg                    ← auto-generated thumbnail
└── output/prototype_samples.json        ← batch output (this notebook writes it)
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  EDIT THESE VARIABLES
# ══════════════════════════════════════════════════════════════════════

VIDEO_PATHS = [
    'data/videos/18 Apr_Genkai Koya-MIXUE.mp4',
    'data/videos/7 Apr_CHOPS-Luckin.mp4',
    "data/videos/Colgate Plax 'Boardroom' - 30 seconds - rishab seth (480p, h264, youtube).mp4",
    'data/videos/Domain 2022 TV Advert  Find your way home  15" - Domain (1080p, h264, youtube).mp4',
    'data/videos/First National Real Estate 2021 TVC 30 seconds - First National Real Estate (1080p, h264, youtube).mp4',
    'data/videos/fishermanfriend_kissme_2017.mp4',
    'data/videos/Head & Shoulder Shampoo Indonesian TVC Myanmar adaptation by SAIL Advertising - Ad Myanmar (480p, h264, youtube).mp4',
    'data/videos/Hiragino Commercial "Waking Up with Home" (30s) _ ひらぎのCM「家と目覚める」（30秒編） - toha film (1080p, h264, youtube).mp4',
    'data/videos/LVIV TVC WingTai Singapore Property 30Secs - MrJackSpaid (720p, h264, youtube).mp4',
    'data/videos/Maggi Mee TVC (30-sec) - Sebastian Sim (720p, h264, youtube).mp4',
    'data/videos/Make Your Meal Moments More Special with Coca-Cola! (SG) - Coca-Cola (720p, h264, youtube).mp4',
    'data/videos/Property development advertisement - Paya Lebar Square - Decibel Lux (360p, h264, youtube).mp4',
    'data/videos/Property Property Property.co.uk 30 second advert - Knight Castle Media (480p, h264, youtube).mp4',
    'data/videos/realestate.com.au 30 second TVC - Development Hub REA Group (1080p, h264, youtube).mp4',
    'data/videos/Your Move Estate Agent TV Ad - YOUR MOVE (480p, h264, youtube).mp4',
    'data/videos/YouTube   Japan Coca Cola Coke Street Live TV commercial 30 sec TVC - Mike O (360p, h264, youtube).mp4',
]

SAMPLES_JSON_PATH = 'data/output/prototype_samples.json'

VIDEO_TITLES = [
    'Mixue — Genkai Koya',
    'Luckin Coffee — CHOPS',
    'Colgate Plax — Boardroom (30s)',
    'Domain — Find Your Way Home (15s)',
    'First National Real Estate — TVC (30s)',
    "Fisherman's Friend — Kiss Me (30s)",
    'Head & Shoulders — Myanmar Adaptation',
    'Hiragino — Waking Up with Home (30s)',
    'Wing Tai Singapore — Property (30s)',
    'Maggi Mee — TVC (30s)',
    'Coca-Cola Singapore — Meal Moments',
    'Paya Lebar Square — Property Ad',
    'Property Property Property.co.uk — Ad (30s)',
    'Realestate.com.au — TVC (30s)',
    'Your Move — Estate Agent TV Ad',
    'Coca-Cola Japan — Street Live (30s)',
]

VIDEO_DESCRIPTIONS = [
    'Mixue ice cream & tea brand spot featuring Genkai Koya.',
    'Luckin Coffee campaign by CHOPS creative.',
    '30-second Colgate Plax mouthwash TVC — boardroom scenario.',
    '15-second Domain Australia property search ad.',
    '30-second First National Real Estate brand TVC (2021).',
    "30-second Fisherman's Friend Singapore — softer brand reinterpretation.",
    'Head & Shoulders Indonesia TVC adapted for Myanmar market.',
    '30-second Japanese real estate commercial — emotional homecoming theme.',
    'Japanese property investment company commercial targeting overseas buyers.',
    '30-second Wing Tai Singapore property development TVC.',
    '30-second Maggi instant noodles Singapore TVC.',
    'Coca-Cola Singapore campaign — meal moments with family.',
    'Paya Lebar Square mixed-use development property advertisement.',
    '30-second UK property portal advertisement.',
    '30-second realestate.com.au Australian property platform TVC.',
    'Your Move UK estate agency television advertisement.',
    '30-second Coca-Cola Japan street vending machine commercial.',
]

# ══════════════════════════════════════════════════════════════════════

# Paths above are relative to the project root (folders are created in the next step)
from pathlib import Path

print(f'Videos to process: {len(VIDEO_PATHS)}')
for i, p in enumerate(VIDEO_PATHS):
    title = VIDEO_TITLES[i] if i < len(VIDEO_TITLES) else p
    print(f'  [{i}] {p} → "{title}"')
print(f'Output: {SAMPLES_JSON_PATH}')


## 2. Load Config & Model

In [ ]:
import os
import sys
import json
import subprocess
import time
from pathlib import Path
import numpy as np
import torch

# ── Locate the project root ──
# Notebooks have no __file__, and Jupyter's CWD depends on where it was
# launched. Walk up from the CWD until config.json turns up, so the ~20 GB
# model download always lands in the project — never in $HOME or /tmp.
def _find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.json').exists():
            return candidate
    raise FileNotFoundError(
        'config.json not found in the CWD or any parent directory. It is committed in\n'
        'the repository: restore it with git checkout config.json.'
    )

PROJECT_DIR = _find_project_root()
CONFIG_PATH = PROJECT_DIR / 'config.json'

# Shared settings come from config.json; the token and AWS identifiers come from .env / the
# environment (see .env.example). Needs `pip install -e .` so neurolens can be imported.
from neurolens.settings import load_settings
CFG = load_settings(PROJECT_DIR)

MODELS_DIR = (PROJECT_DIR / CFG['paths']['models']).resolve()
DATA_DIR   = (PROJECT_DIR / CFG['paths']['data']).resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'data/videos').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'data/output').mkdir(parents=True, exist_ok=True)

HF_TOKEN = CFG.get('hf_token', '')
assert HF_TOKEN, 'Missing HF_TOKEN. Put it in .env (see .env.example).'

# ── Set all HuggingFace env vars ──
# huggingface_hub freezes these into module-level constants on FIRST import.
# Run this cell before any cell that imports huggingface_hub / transformers /
# tribev2, or the weights silently go to ~/.cache/huggingface instead.
if 'huggingface_hub' in sys.modules:
    print('⚠️  huggingface_hub is already imported in this kernel.')
    print('   Restart the kernel (Kernel ▸ Restart) and run this cell first,')
    print('   or downloads land in ~/.cache/huggingface, not ./models.')
    print()

os.environ['HF_TOKEN']                 = HF_TOKEN
os.environ['HF_HOME']                 = str(MODELS_DIR)
os.environ['HF_HUB_CACHE']            = str(MODELS_DIR / 'hub')
os.environ['HUGGINGFACE_HUB_CACHE']   = str(MODELS_DIR / 'hub')      # legacy alias
os.environ['HF_ASSETS_CACHE']         = str(MODELS_DIR / 'assets')
os.environ['HF_DATASETS_CACHE']       = str(MODELS_DIR / 'datasets')
os.environ['HF_XET_CACHE']            = str(MODELS_DIR / 'xet')      # hf_xet dedup cache
os.environ['TORCH_HOME']              = str(MODELS_DIR / 'torch')    # torch.hub weights
os.environ['NILEARN_DATA']            = str(DATA_DIR / 'nilearn')
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = str(CFG.get('hf_download_timeout', 300))
os.environ['HF_HUB_HTTP_TIMEOUT']     = str(CFG.get('hf_download_timeout', 300))

print(f'Project root: {PROJECT_DIR}')
print(f'Models / HF:  {MODELS_DIR}')
print()

print('Loading TRIBE v2...')
from tribev2.demo_utils import TribeModel
model = TribeModel.from_pretrained(CFG['model']['repo_id'], cache_folder=str(MODELS_DIR))
print('✅ Model loaded.')

print('Loading Destrieux atlas...')
from nilearn import datasets as nl_datasets
destrieux = nl_datasets.fetch_atlas_surf_destrieux()
labels_lh = np.array(destrieux['map_left'])
labels_rh = np.array(destrieux['map_right'])
label_names = destrieux['labels']
labels_full = np.concatenate([labels_lh, labels_rh])
print('✅ Atlas loaded.')


## 3. Build ROI Masks

In [ ]:
ROI_LABEL_MAP = {
    'ffa_faces': ['G_oc-temp_lat-fusifor'],
    'eba_bodies': ['S_oc-temp_lat', 'G_temporal_inf'],
    'ppa_scenes': ['G_oc-temp_med-Parahip'],
    'sts_social': ['S_temporal_sup'],
    'auditory': ['G_temp_sup-G_T_transv', 'G_temp_sup-Plan_tempo'],
}

roi_masks = {}
for roi_name, target_labels in ROI_LABEL_MAP.items():
    mask = np.zeros(len(labels_full), dtype=bool)
    for target in target_labels:
        for i, name in enumerate(label_names):
            if target in name:
                mask |= (labels_full == i)
    roi_masks[roi_name] = mask
    print(f'  {roi_name}: {mask.sum()} vertices')

def normalize_01(arr):
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-8: return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

print('✅ ROI masks ready.')


## 4. Processing Functions

In [ ]:
def process_video(video_path):
    """Run full pipeline on a single video. Returns timesteps list."""
    video_path = Path(video_path)
    print(f'\n{"═" * 60}')
    print(f'Processing: {video_path.name}')
    print(f'{"═" * 60}')
    
    # Full inference (video + audio)
    print('  [1/4] Building events (full video)...')
    t0 = time.time()
    events_full = model.get_events_dataframe(video_path=str(video_path))
    print('  [2/4] Running predict (full)...')
    preds_full, _ = model.predict(events=events_full)
    preds_full = np.asarray(preds_full)
    print(f'         Shape: {preds_full.shape} in {time.time()-t0:.1f}s')
    
    # Strip audio
    noaudio_path = Path('/tmp') / f'{video_path.stem}.noaudio.mp4'
    print('  [3/4] Stripping audio + running predict (video-only)...')
    subprocess.run(['ffmpeg', '-y', '-i', str(video_path), '-an', '-c:v', 'copy', str(noaudio_path)],
                   capture_output=True, text=True)
    events_noaudio = model.get_events_dataframe(video_path=str(noaudio_path))
    preds_noaudio, _ = model.predict(events=events_noaudio)
    preds_noaudio = np.asarray(preds_noaudio)
    noaudio_path.unlink(missing_ok=True)
    
    # Extract engagement
    print('  [4/4] Extracting ROI engagement...')
    T = preds_full.shape[0]
    roi_ts = {}
    for roi_name, mask in roi_masks.items():
        roi_ts[roi_name] = np.abs(preds_full[:, mask]).mean(axis=1)
    roi_ts['engagement_overall'] = np.mean([roi_ts[k] for k in ROI_LABEL_MAP], axis=0)
    roi_normed = {name: normalize_01(ts) for name, ts in roi_ts.items()}
    
    aud_mask = roi_masks['auditory']
    aud_full = normalize_01(np.abs(preds_full[:, aud_mask]).mean(axis=1))
    aud_noaudio = normalize_01(np.abs(preds_noaudio[:, aud_mask]).mean(axis=1))
    min_len = min(len(aud_full), len(aud_noaudio))
    
    timesteps = []
    for t in range(T):
        step = {
            't': t,
            'engagement_overall': round(float(roi_normed['engagement_overall'][t]), 4),
            'regions': {
                'ffa_faces': round(float(roi_normed['ffa_faces'][t]), 4),
                'eba_bodies': round(float(roi_normed['eba_bodies'][t]), 4),
                'ppa_scenes': round(float(roi_normed['ppa_scenes'][t]), 4),
                'sts_social': round(float(roi_normed['sts_social'][t]), 4),
                'auditory': round(float(roi_normed['auditory'][t]), 4),
            },
            'auditory_with_audio': round(float(aud_full[t]), 4) if t < min_len else None,
            'auditory_without_audio': round(float(aud_noaudio[t]), 4) if t < min_len else None,
        }
        timesteps.append(step)
    
    elapsed = time.time() - t0
    print(f'  ✅ Done — {T} timesteps in {elapsed:.1f}s')
    return timesteps, T


def generate_thumbnail(video_path, output_dir):
    """Extract a thumbnail at the 3-second mark using ffmpeg."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    thumb_path = output_dir / 'thumb.jpg'
    result = subprocess.run([
        'ffmpeg', '-y', '-i', str(video_path),
        '-ss', '3', '-vframes', '1',
        '-vf', 'scale=280:-1',
        str(thumb_path)
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  Thumbnail: {thumb_path}')
        return thumb_path
    else:
        print(f'  ⚠️ Thumbnail failed: {result.stderr[:100]}')
        return None

print('✅ Functions ready.')


## 5. Batch Process All Videos

Loads existing `prototype_samples.json` (if it exists), skips videos that are already processed,  
and appends new results. Re-running this notebook with additional videos in `VIDEO_PATHS`  
will only process the new ones.

In [ ]:
# Load existing prototype_samples.json if it exists
output_path = PROJECT_DIR / SAMPLES_JSON_PATH
existing_samples = []
existing_ids = set()

if output_path.exists():
    with open(output_path) as f:
        existing_data = json.load(f)
    existing_samples = existing_data.get('samples', [])
    existing_ids = {s['id'] for s in existing_samples}
    print(f'Loaded existing prototype_samples.json: {len(existing_samples)} videos already processed')
    for s in existing_samples:
        print(f'  ✓ {s["id"]} — {s["title"]}')
else:
    print('No existing prototype_samples.json — starting fresh.')

# Process only new videos
new_samples = []
skipped = 0
total_start = time.time()

for idx, vpath in enumerate(VIDEO_PATHS):
    vpath = PROJECT_DIR / vpath
    video_id = vpath.stem
    
    if not vpath.exists():
        print(f'⚠️  Skipping — file not found: {vpath}')
        continue
    
    if video_id in existing_ids:
        print(f'⏭️  Skipping — already processed: {video_id}')
        skipped += 1
        continue
    
    # Process
    timesteps, duration = process_video(vpath)
    
    # Per-video output directory: data/output/[video_stem]/
    video_output_dir = PROJECT_DIR / 'data/output' / video_id
    generate_thumbnail(vpath, video_output_dir)
    
    # Title and description
    title = VIDEO_TITLES[idx] if idx < len(VIDEO_TITLES) else vpath.stem.replace('_', ' ').title()
    desc = VIDEO_DESCRIPTIONS[idx] if idx < len(VIDEO_DESCRIPTIONS) else ''
    
    # Paths stored relative to data/
    sample = {
        'id': video_id,
        'title': title,
        'description': desc,
        'video_file': f'videos/{vpath.name}',
        'thumbnail': f'output/{video_id}/thumb.jpg',
        'duration_seconds': duration,
        'timesteps': timesteps,
    }
    new_samples.append(sample)

total_elapsed = time.time() - total_start
print(f'\n{"═" * 60}')
print(f'Batch complete:')
print(f'  Existing: {len(existing_samples)} videos (kept)')
print(f'  Skipped:  {skipped} (already in prototype_samples.json)')
print(f'  New:      {len(new_samples)} videos processed in {total_elapsed:.1f}s')
print(f'  Total:    {len(existing_samples) + len(new_samples)} videos')
print(f'{"═" * 60}')


## 6. Write prototype_samples.json

In [ ]:
# Merge existing samples with new ones
all_samples = existing_samples + new_samples

output = {'samples': all_samples}

output_path = PROJECT_DIR / SAMPLES_JSON_PATH
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

file_size = output_path.stat().st_size / 1024
print(f'✅ Written to {output_path} ({file_size:.1f} KB)')
print(f'   {len(all_samples)} total samples:')
for s in all_samples:
    is_new = s['id'] in {ns['id'] for ns in new_samples}
    tag = '  NEW' if is_new else '     '
    print(f'   {tag} {s["title"]} ({s["duration_seconds"]}s, {len(s["timesteps"])} timesteps)')
